# 01 — Ingest, QC and schema discovery

**Frangieh et al. 2021 Perturb-CITE-seq** — ~218k patient-derived melanoma
cells, 248 CRISPR-KO targets, three environments (control / IFN-γ / TIL
co-culture), RNA + 20-plex ADT.

**Goal of this notebook:** find out what is actually in these files, correct
`config.yaml` to match, and produce the power table that determines what the
rest of the project is allowed to claim.

Nothing downstream should run until `check_schema()` passes.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg = load_config()
panels = load_panels()
P = paths(cfg)
SEED = set_seed(cfg)
apply_style(cfg)

sc.settings.verbosity = 1
print(f"repo: {P.root}")
print(f"seed: {SEED}")


## 1. Fetch

Idempotent — skips files already on disk (~3-5 GB).

In [ ]:
from src.data_io import fetch_raw
files = fetch_raw(cfg)
files

## 2. Schema discovery

**Run this before writing any analysis code.** The `schema:` block in
`config/config.yaml` is an assumption about scPerturb's harmonised column
names. Correct it here, then everything downstream inherits the fix.

In [ ]:
from src.data_io import load_rna, load_protein, describe_schema

rna = load_rna(cfg)
rna_schema = describe_schema(rna, "RNA")

In [ ]:
adt = load_protein(cfg)
adt_schema = describe_schema(adt, "ADT")
print("\nADT panel:", list(adt.var_names))

### Reconcile

Edit `config/config.yaml` → `schema:` now if the printed columns differ from
what is declared, then re-run the cell below. It raises rather than warns —
a loud failure here is far cheaper than a silent `KeyError` forty cells on.

In [ ]:
from src.data_io import check_schema
check_schema(rna, cfg)

### Write the observed ADT panel

`panels.yaml` ships with an empty `adt_to_rna` stub on purpose — guessing a
20-plex panel would be worse than leaving it blank. Dump the real feature
names, then map them to gene symbols **by hand**, minding many-to-one cases
(an MHC-I antibody may detect HLA-A, -B and -C).

In [ ]:
import yaml
observed = {
    "adt": {
        "observed_features": list(map(str, adt.var_names)),
        "adt_to_rna": {f: [] for f in map(str, adt.var_names)},
    }
}
out = P.root / "config" / "panels_observed.yaml"
out.write_text(yaml.safe_dump(observed, sort_keys=False))
print(f"wrote {out}\n-> fill in adt_to_rna, merge into config/panels.yaml, commit")

## 3. Align modalities

The two h5ads are distributed separately and are not guaranteed to contain the
same cells in the same order.

In [ ]:
from src.data_io import align_modalities
rna, adt = align_modalities(rna, adt)

## 4. The power table

Cells per perturbation × condition. **This single figure sets the ceiling on
what the project can claim.** An arm with 12 cells will not support a
context-dependence call regardless of what any p-value says.

Note the filter is applied *per condition*, not globally: a perturbation can
be well powered in control and underpowered in co-culture (because those cells
were killed), and that asymmetry is itself informative.

In [ ]:
from src.pseudobulk import group_sizes

sizes = group_sizes(rna, cfg)
print(f"{sizes.shape[0]} perturbations x {sizes.shape[1]} conditions")
sizes.describe()

In [ ]:
import seaborn as sns

min_n = cfg["qc"]["min_cells_per_perturbation_per_condition"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(np.log10(sizes + 1), cmap=cfg["plotting"]["cmap_sequential"],
            ax=axes[0], cbar_kws={"label": "log10(cells + 1)"}, yticklabels=False)
axes[0].set_title("Cells per perturbation x condition")

passed = (sizes >= min_n).sum()
axes[1].bar(passed.index, passed.values,
            color=[condition_palette(cfg).get(c, "#888") for c in passed.index])
axes[1].axhline(len(sizes), ls="--", c="k", lw=1, label="all perturbations")
axes[1].set_ylabel(f"perturbations with >= {min_n} cells")
axes[1].legend()
plt.tight_layout()
savefig(fig, "01_power_table", cfg)

### TODO

- [ ] How many perturbations are powered in **all three** conditions? Only those support a clean context-dependence call.
- [ ] Is the co-culture arm systematically thinner? If so, that is selection, and it previews nb05.

In [ ]:
powered_everywhere = (sizes >= min_n).all(axis=1)
print(f"powered in all conditions: {powered_everywhere.sum()} / {len(sizes)}")
sizes.loc[~powered_everywhere].head(20)

## 5. Cell-level QC

scPerturb already applied a uniform QC pass, so thresholds here are
deliberately permissive. Tighten only with a plot that justifies it.

Expect the co-culture arm to look different — those cells are under attack.
Do not "fix" that; it is the biology.

In [ ]:
s = cfg["schema"]["obs"]
qc_cols = [s["n_counts"], s["n_genes"], s["percent_mito"]]
qc_cols = [c for c in qc_cols if c in rna.obs.columns]

fig, axes = plt.subplots(1, len(qc_cols), figsize=(4.5 * len(qc_cols), 4))
for ax, col in zip(np.atleast_1d(axes), qc_cols):
    sns.violinplot(data=rna.obs, x=s["condition"], y=col, ax=ax,
                   palette=condition_palette(cfg), cut=0)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
savefig(fig, "01_qc_by_condition", cfg)

### Guide multiplicity

Expect a low-MOI design (one guide per cell). Cells with more than one assigned guide confound every downstream contrast.

In [ ]:
npert_col = s.get("n_perturbations")
if npert_col in rna.obs.columns:
    print(rna.obs[npert_col].value_counts().sort_index())
    if cfg["qc"]["drop_multi_guide_cells"]:
        keep = rna.obs[npert_col].astype(float) <= 1
        print(f"dropping {(~keep).sum():,} multi-guide cells")
        rna, adt = rna[keep].copy(), adt[keep].copy()
else:
    print(f"[warn] no column {npert_col!r}; check guide assignment manually")

## 6. Apply filters and save

In [ ]:
# TODO: apply gene, mito and per-condition perturbation-count filters,
# then write the paired object.
#
#   import mudata as md
#   mdata = md.MuData({"rna": rna, "adt": adt})
#   mdata.write(P.data_interim / "frangieh_qc.h5mu")

print("[stub] implement filtering + write to data/interim/frangieh_qc.h5mu")

In [ ]:
import session_info
session_info.show()